In [ ]:
from pyspark.sql import SparkSession, functions as F
from IPython.display import display

spark = (SparkSession.builder
    .appName("olist-02-exploration")
    .config("spark.sql.warehouse.dir", "hdfs:///user/hive/warehouse")
    .enableHiveSupport()
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

RAW    = "hdfs:///data/olist/raw"
BUCKET = "gs://olist-sujinan-2026"
DB     = "olist"

def show(df, n=20):
    display(df.limit(n).toPandas())

def pdf(df, n=1000):
    return df.limit(n).toPandas()

print("warehouse:", spark.conf.get("spark.sql.warehouse.dir"))

# Detect missing values function

In [ ]:
names = ["customers", "orders", "order_items", "payments", "reviews",
         "products", "sellers", "geolocation", "categories"]
dfs = {n: spark.table(f"{DB}.bronze_{n}") for n in names}


def missing_report(df, name):
    """Which columns have null values, how many rows, and what percentage"""
    total = df.count()

    exprs = [F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns]
    counts = df.select(*exprs).first().asDict()

    rows = [
        (name, col, n, round(100 * n / total, 2))
        for col, n in counts.items() if n > 0
    ]
    return spark.createDataFrame(
        rows, "dataset string, column string, null_count long, null_pct double"
    )


reports = [missing_report(df, name) for name, df in dfs.items()]
all_missing = reports[0]
for r in reports[1:]:
    all_missing = all_missing.union(r)

show(all_missing.orderBy(F.desc("null_pct")), 40)

# Check for duplicates

In [ ]:
def duplicate_report(df, key):
    return (df.groupBy(key)
              .count()
              .filter(F.col("count") > 1)
              .orderBy(F.desc("count")))

show(duplicate_report(dfs["customers"], "customer_id")) 
show(duplicate_report(dfs["orders"],    "order_id"))    
show(duplicate_report(dfs["geolocation"], "geolocation_zip_code_prefix"))

# EDA 

In [ ]:
# Where are customers – For warehouse planning
show(dfs["customers"].groupBy("customer_state").count()
     .orderBy(F.desc("count")))

In [ ]:
# Order Status Breakdown — If Delivered isn't the top status, the business has a problem.
show(dfs["orders"].groupBy("order_status").count()
     .orderBy(F.desc("count")))

In [ ]:
# Popular Payment Methods — Helping you decide which payment gateway to invest in
show(dfs["payments"].groupBy("payment_type").count()
     .orderBy(F.desc("count")))

In [ ]:
# Top 10 Highest-Earning Products
show(dfs["order_items"].groupBy("product_id")
     .agg(F.round(F.sum("price"), 2).alias("total_sales"))
     .orderBy(F.desc("total_sales")), 10)

# Delivery time analysis

In [ ]:
delivery = (dfs["orders"]
    .withColumn(
        "delivery_days",
        F.datediff("order_delivered_customer_date", "order_purchase_timestamp")
    )
    .select("order_id", "order_status",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "delivery_days")
)

# Slowest-delivered orders — should be discussed with the logistics team.
show(delivery.orderBy(F.desc("delivery_days")))

In [ ]:
# Distribution Overview
show(delivery.select("delivery_days").summary())